In [1]:
!pip install -q torch torchaudio transformers datasets peft accelerate rich soundfile jiwer

import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 59.7 MB/s eta 0:00:00
PyTorch: 2.9.0+cu126
CUDA: True
GPU: Tesla T4


In [2]:
import zipfile
import os

zip_name = '/content/approach 7.zip'  # adjust if different
with zipfile.ZipFile(zip_name, 'r') as z:
    z.extractall('/content/')

ROOT = '/content/approach 7'
os.chdir(ROOT)
!ls -la

total 156
drwxr-xr-x 5 root root  4096 Jan 29 14:42 .
drwxr-xr-x 1 root root  4096 Jan 29 14:42 ..
drwxr-xr-x 5 root root  4096 Jan 29 14:42 audio
drwxr-xr-x 6 root root  4096 Jan 29 14:42 data
-rw-r--r-- 1 root root 21790 Jan 29 14:42 evaluate_whisper.py
-rw-r--r-- 1 root root 39396 Jan 29 14:42 finetune_whisper.py
-rw-r--r-- 1 root root 22540 Jan 29 14:42 generate_addresses.py
-rw-r--r-- 1 root root  9716 Jan 29 14:42 generate_audio.py
-rw-r--r-- 1 root root 19502 Jan 29 14:42 generate_sentences.py
drwxr-xr-x 2 root root  4096 Jan 29 14:42 models
-rw-r--r-- 1 root root  4435 Jan 29 14:42 prepare_whisper_dataset.py
-rw-r--r-- 1 root root  3285 Jan 29 14:42 rebuild_audio_metadata.py
-rw-r--r-- 1 root root   174 Jan 29 14:42 requirements.txt


In [3]:
import os
ROOT = '/content/approach 7'
os.chdir(ROOT)

!sed -i 's|/Users/web1havv/vaibhav site/public/napiers/bolna_task/approach 7|/content/approach 7|g' audio/train/*_metadata.jsonl 2>/dev/null
!sed -i 's|/Users/web1havv/vaibhav site/public/napiers/bolna_task/approach 7|/content/approach 7|g' audio/test/*_metadata.jsonl 2>/dev/null
!sed -i 's|/Users/web1havv/vaibhav site/public/napiers/bolna_task/approach 7|/content/approach 7|g' audio/eval/*_metadata.jsonl 2>/dev/null
print("Paths fixed!")

Paths fixed!


In [4]:
!python prepare_whisper_dataset.py
!wc -l data/whisper_dataset/*.jsonl

Prepared Whisper dataset at /content/approach 7/data/whisper_dataset with 4669 
train, 1000 test, 1002 eval examples.
   1002 data/whisper_dataset/eval.jsonl
   1000 data/whisper_dataset/test.jsonl
   4669 data/whisper_dataset/train.jsonl
   6671 total


In [5]:
!python finetune_whisper.py \
    --phoneme-epochs 0 \
    --canonical-epochs 3 \
    --lora-rank 24 \
    --focal-gamma 2.0 \
    --label-smoothing 0.1 \
    --sequence-bonus 0.5 \
    --alpha-start 1.5 \
    --alpha-end 2.0 \
    --specaugment \
    --bf16 \
    --per-device-train-batch-size 8 \
    --gradient-accumulation-steps 2 \
    --dataloader-num-workers 0 \
    --max-train-samples 2000

2026-01-29 14:43:02.915463: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769697782.935259    2917 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769697782.941107    2917 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769697782.957377    2917 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697782.957402    2917 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769697782.957406    2917 computation_placer.cc:177] computation placer alr

In [6]:
!python evaluate_whisper.py --eval-split test --max-samples 500

2026-01-29 15:01:35.193101: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1769698895.212578    7704 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1769698895.218555    7704 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1769698895.233804    7704 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769698895.233826    7704 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1769698895.233830    7704 computation_placer.cc:177] computation placer alr

In [7]:
import shutil
from google.colab import files
ROOT = '/content/approach 7'
shutil.make_archive('/content/whisper_lora_approach7', 'zip', f'{ROOT}/models/whisper_base_bangalore_lora')
files.download('/content/whisper_lora_approach7.zip')

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>